# MALTO Text Authorship Detection
## Run: `2026-03-16_221821_run` | Model: `stacking_lgbm` | LB: **0.92422**

**6-class classification**: Human (0), DeepSeek (1), Grok (2), Claude (3), Gemini (4), ChatGPT (5)  
**Metric**: Macro F1  

This notebook **loads the pre-trained model** from the experiment run. No re-training needed.

### Architecture
```
Raw text
  → Preprocessor (unicode normalization, whitespace)
  → FeatureUnion (7 branches, ~120k features)
      ├─ word_tfidf      (50k, ngram 1-2)
      ├─ char_tfidf      (50k, ngram 2-6, char_wb)
      ├─ stylometric     (43 hand-crafted features)
      └─ function_word   (5k, ngram 1-2)
  → StackingClassifier
      ├─ Base 1: TwoStageClassifier (LR balanced + binary DS/Grok)
      ├─ Base 2: TfidfMLPClassifier (SVD-500 + MLP 512-256)
      ├─ Base 3: LGBMTfidfClassifier (SVD-300 + LightGBM)
      └─ Meta:   LogisticRegression (C=0.1, balanced)
  → Thresholds: [2.25, 0.75, 1.0, 1.0, 2.5, 1.75]
  → DS/Grok pair threshold: 0.55
```

### CV Results (this run)
| Model | CV F1 | OOF F1 |
|---|---|---|
| stacking_lgbm | **0.9393** | **0.9397** |
| ensemble_lgbm | 0.9334 | 0.9338 |
| ensemble_mlp | 0.9305 | 0.9314 |
| two_stage_top2 | 0.9303 | 0.9313 |

**Leaderboard**: 0.92422

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lightgbm>=4.0', 'xgboost>=2.0', 'scikit-learn>=1.3',
    'numpy', 'pandas', 'scipy', 'joblib'
], check=True)
print('Dependencies OK.')

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import math
import re
import string
import unicodedata
import json
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
print('Imports OK.')

## 1. Paths

**On Kaggle**: Upload `best_model.joblib` and `thresholds.json` as a dataset named `malto-model`.  
**Locally**: Points directly to the experiment directory.

In [ ]:
import os

if os.path.exists('/kaggle/input'):
    # ── Kaggle ────────────────────────────────────────────────────────────────
    # Data: upload train.csv / test.csv as dataset 'malto-hackathon'
    DATA_DIR   = Path('/kaggle/input/malto-hackathon')
    # Model: upload best_model.joblib + thresholds.json as dataset 'malto-model'
    MODEL_DIR  = Path('/kaggle/input/malto-model')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    # ── Local — experiment run 2026-03-16_221821_run ──────────────────────────
    REPO_ROOT  = Path(r'D:/hachaton/text-authorship-detection')
    DATA_DIR   = REPO_ROOT / 'data'
    MODEL_DIR  = REPO_ROOT / 'artifacts/experiments/2026-03-16_221821_run'
    OUTPUT_DIR = REPO_ROOT / 'artifacts/submissions'

TRAIN_FILE      = DATA_DIR  / 'train.csv'
TEST_FILE       = DATA_DIR  / 'test.csv'
SAMPLE_FILE     = DATA_DIR  / 'sample_submission.csv'
MODEL_FILE      = MODEL_DIR / 'best_model.joblib'
THRESHOLDS_FILE = MODEL_DIR / 'thresholds.json'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Train      : {TRAIN_FILE}   exists={TRAIN_FILE.exists()}')
print(f'Test       : {TEST_FILE}    exists={TEST_FILE.exists()}')
print(f'Model      : {MODEL_FILE}   exists={MODEL_FILE.exists()}')
print(f'Thresholds : {THRESHOLDS_FILE}  exists={THRESHOLDS_FILE.exists()}')

## 2. Custom Class Definitions

These classes must be defined **before** loading the joblib model, because joblib
deserializes by reference to the class definitions. They are identical to the
classes used during training.

In [ ]:
# ── Preprocessor ──────────────────────────────────────────────────────────────
class Preprocessor(BaseEstimator, TransformerMixin):
    """
    Configurable text preprocessor. Preserves stylistic signals:
    punctuation, capitalization, newlines are kept (they are discriminative).
    Only cleans unicode, whitespace, repeated spaces.
    """
    def __init__(self, normalize_unicode=True, strip_whitespace=True,
                 remove_repeated_spaces=True, lowercase=False,
                 remove_punctuation=False, remove_numbers=False):
        self.normalize_unicode      = normalize_unicode
        self.strip_whitespace       = strip_whitespace
        self.remove_repeated_spaces = remove_repeated_spaces
        self.lowercase              = lowercase
        self.remove_punctuation     = remove_punctuation
        self.remove_numbers         = remove_numbers

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return [self._clean(text) for text in X]

    def _clean(self, text):
        if not isinstance(text, str):
            text = str(text) if text is not None else ''
        if self.normalize_unicode:
            text = unicodedata.normalize('NFC', text)
        if self.strip_whitespace:
            text = text.strip()
        if self.remove_repeated_spaces:
            text = re.sub(r' {2,}', ' ', text)
            text = re.sub(r'\r\n?', '\n', text)
            text = re.sub(r'\t+', ' ', text)
            text = re.sub(r' {2,}', ' ', text)
        if self.lowercase:
            text = text.lower()
        if self.remove_punctuation:
            text = re.sub(r'[^\w\s]', '', text)
        if self.remove_numbers:
            text = re.sub(r'\d+', '<NUM>', text)
        return text

print('Preprocessor defined.')

In [ ]:
# ── Function-word vocabulary (for FunctionWordAnalyzer) ───────────────────────
FUNCTION_WORDS = [
    'the','a','an','this','that','these','those','my','your','his','her','its','our','their',
    'of','in','on','at','to','for','with','by','from','about','as','into','through','during',
    'before','after','above','below','between','among','under','over','within','without','upon',
    'and','but','or','so','yet',
    'although','because','since','while','if','unless','until','when','where','whether','though',
    'is','are','was','were','be','been','being',
    'have','has','had','do','does','did',
    'will','would','shall','should','may','might','must','can','could',
    'i','you','he','she','it','we','they','me','him','us','them',
    'however','therefore','moreover','furthermore','additionally',
    'nevertheless','nonetheless','thus','hence','consequently',
    'accordingly','whereas','meanwhile','subsequently','overall',
    'also','too','either','neither',
    'very','quite','rather','somewhat','even','just','only',
    'indeed','certainly','particularly','especially','generally',
    'actually','basically','literally','simply',
    'all','some','any','each','every','many','much','few','little',
    'more','most','other','another','same','different','various','several',
    'which','who','whom','whose','what','how','why',
    'not','no','never','nor',
]


# ── StyleometricTransformer ───────────────────────────────────────────────────
class StyleometricTransformer(BaseEstimator, TransformerMixin):
    """43 hand-crafted stylometric features (version used in run 2026-03-16)."""
    _NUM_RE  = re.compile(r'^\s*\d+[\.)].+', re.MULTILINE)
    _HEAD_RE = re.compile(r'^#{1,6}\s', re.MULTILINE)
    _BOLD_RE = re.compile(r'\*\*')
    _ITAL_RE = re.compile(r'(?<!\*)\*(?!\*)')
    _CODE_RE = re.compile(r'`')
    _LINK_RE = re.compile(r'\[.+?\]\(.+?\)')
    _ELIP_RE = re.compile(r'\.\.\.')
    _CAPS_RE = re.compile(r'\b[A-Z]{2,}\b')
    _TRANS_RE = re.compile(
        r'\b(however|therefore|moreover|furthermore|additionally|consequently|'
        r'nevertheless|nonetheless|whereas|thus|hence|accordingly|subsequently|'
        r'in conclusion|in summary|for example|for instance|in particular)\b',
        re.IGNORECASE
    )
    _HEDGE_RE = re.compile(
        r'\b(may|might|could|possibly|perhaps|likely|suggests|suggest|'
        r'appears|appear|seems|seem|potentially|presumably|arguably|'
        r'apparently|typically|usually|often)\b',
        re.IGNORECASE
    )

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return np.array([self._f(t) for t in X], dtype=np.float32)

    def _f(self, text):
        """Extract 43 features. NOTE: This run used 43 features (not 49)."""
        if not text or not isinstance(text, str):
            return [0.0] * 43
        NL2 = '\n\n'
        NL1 = '\n'
        ch   = len(text)
        words = text.split(); nw = max(len(words), 1)
        sents = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
        ns    = max(len(sents), 1)
        paras = [p.strip() for p in text.split(NL2) if p.strip()]
        np_   = max(len(paras), 1)
        lines = text.split(NL1); nl_ = max(len(lines), 1)

        wl  = [len(w.strip(string.punctuation)) for w in words if w.strip(string.punctuation)]
        awl = float(np.mean(wl)) if wl else 0.0
        ttr = len(set(w.lower().strip(string.punctuation) for w in words)) / nw
        lwr = sum(1 for l in wl if l > 6) / nw
        swc = [len(s.split()) for s in sents]
        vss = sum(1 for c in swc if c < 5) / ns
        vls = sum(1 for c in swc if c > 30) / ns
        apc = float(np.mean([len(p) for p in paras])) if paras else 0.0

        cm  = text.count(',') / nw
        per = text.count('.') / nw
        ex  = text.count('!') / nw
        qu  = text.count('?') / nw
        co  = text.count(':') / nw
        se  = text.count(';') / nw
        qt  = (text.count('"') + text.count("'") +
               text.count('\u201c') + text.count('\u201d')) / nw
        pa  = (text.count('(') + text.count(')')) / nw
        el  = len(self._ELIP_RE.findall(text)) / nw
        da  = (text.count('-') + text.count('\u2014')) / nw

        alpha = [c for c in text if c.isalpha()]
        na    = max(len(alpha), 1)
        ucr   = sum(1 for c in alpha if c.isupper()) / na
        dr    = sum(1 for c in text if c.isdigit()) / max(ch, 1)
        cwr   = len(self._CAPS_RE.findall(text)) / nw
        nlr   = text.count(NL1) / max(ch, 1)

        br  = sum(1 for l in lines if l.lstrip().startswith(('-','*','\u2022'))) / nl_
        nr  = len(self._NUM_RE.findall(text)) / nl_
        hr  = len(self._HEAD_RE.findall(text)) / nl_
        cr  = len(self._CODE_RE.findall(text)) / max(ch, 1) * 100
        bor = len(self._BOLD_RE.findall(text)) / max(ch, 1) * 100
        ir  = len(self._ITAL_RE.findall(text)) / max(ch, 1) * 100
        lkr = len(self._LINK_RE.findall(text)) / nw
        isr = sum(1 for s in sents if re.match(r'^I\s', s.strip())) / ns

        punct_chars   = [c for c in text if c in string.punctuation]
        punct_variety = len(set(punct_chars)) / max(len(punct_chars), 1)
        sent_cv = float(np.std(swc) / max(np.mean(swc), 1)) if len(swc) > 1 else 0.0
        trans_rate       = len(self._TRANS_RE.findall(text)) / ns
        fsw              = float(len(sents[0].split())) if sents else 0.0
        cap_words        = sum(1 for w in words if w and w[0].isupper())
        proper_noun_dens = max(cap_words - ns, 0) / nw
        hedge_rate       = len(self._HEDGE_RE.findall(text)) / ns
        qps              = text.count('?') / ns
        sent_range       = float(max(swc) - min(swc)) if len(swc) > 1 else 0.0
        text_len_log     = math.log10(max(ch, 1))
        starts_with_the  = 1.0 if text.startswith('The ') else 0.0
        clause_per_sent  = text.count(',') / ns

        return [
            ch, nw, ns, np_, awl, nw/ns, nw/np_, ttr, lwr,
            vss, vls, apc, cm, per, ex, qu, co, se, qt, pa, el, da,
            ucr, dr, cwr, nlr, br, nr, hr, cr, bor, ir, lkr, isr,
            punct_variety, sent_cv, trans_rate, fsw, proper_noun_dens,
            hedge_rate, qps, sent_range, text_len_log,
        ]


class DenseToSparse(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X, y=None):
        return sp.csr_matrix(X if isinstance(X, np.ndarray) else np.array(X))


class StyleometricPipeline(BaseEstimator, TransformerMixin):
    """StyleometricTransformer + MaxAbsScaler + sparse conversion."""
    def __init__(self):
        self.extractor = StyleometricTransformer()
        self.scaler    = MaxAbsScaler()

    def fit(self, X, y=None):
        self.scaler.fit(self.extractor.transform(X))
        return self

    def transform(self, X, y=None):
        return sp.csr_matrix(self.scaler.transform(self.extractor.transform(X)))


class FunctionWordAnalyzer:
    """Callable: extracts n-grams of function words only. Picklable."""
    def __init__(self, ngram_range=(1, 2)):
        self.ngram_range = ngram_range
        self._fw_set     = frozenset(FUNCTION_WORDS)
        self._tok_re     = re.compile(r'(?u)\b\w+\b')

    def __call__(self, text):
        tokens = [t for t in self._tok_re.findall(text.lower()) if t in self._fw_set]
        min_n, max_n = self.ngram_range
        grams = []
        for n in range(min_n, max_n + 1):
            for i in range(len(tokens) - n + 1):
                grams.append(' '.join(tokens[i:i+n]))
        return grams


print('Feature engineering classes defined.')

In [ ]:
# ── Model classes (required for joblib deserialization) ───────────────────────

class TfidfMLPClassifier(ClassifierMixin, BaseEstimator):
    """TruncatedSVD(n) → MLPClassifier with balanced sample weights."""
    _estimator_type = 'classifier'

    def __init__(self, n_svd_components=500, hidden_layer_sizes=(512,256),
                 activation='relu', alpha=0.01, max_iter=300,
                 early_stopping=True, validation_fraction=0.1,
                 random_state=42, deepseek_boost_factor=1.0, verbose=False):
        self.n_svd_components     = n_svd_components
        self.hidden_layer_sizes   = hidden_layer_sizes
        self.activation           = activation
        self.alpha                = alpha
        self.max_iter             = max_iter
        self.early_stopping       = early_stopping
        self.validation_fraction  = validation_fraction
        self.random_state         = random_state
        self.deepseek_boost_factor = deepseek_boost_factor
        self.verbose              = verbose

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_components  = min(self.n_svd_components, X.shape[1]-1, X.shape[0]-1)
        self.svd_     = TruncatedSVD(n_components=n_components, random_state=self.random_state)
        X_dense       = self.svd_.fit_transform(X)
        sample_weights = np.asarray(compute_sample_weight('balanced', y), dtype=np.float64)
        if self.deepseek_boost_factor != 1.0:
            sample_weights[np.asarray(y) == 1] *= float(self.deepseek_boost_factor)
        self.mlp_ = MLPClassifier(
            hidden_layer_sizes=self.hidden_layer_sizes, activation=self.activation,
            solver='adam', alpha=self.alpha, max_iter=self.max_iter,
            early_stopping=self.early_stopping, validation_fraction=self.validation_fraction,
            random_state=self.random_state, n_iter_no_change=15, verbose=self.verbose,
        )
        self.mlp_.fit(X_dense, y, sample_weight=sample_weights)
        return self

    def predict(self, X):       return self.mlp_.predict(self.svd_.transform(X))
    def predict_proba(self, X): return self.mlp_.predict_proba(self.svd_.transform(X))


class TwoStageClassifier(ClassifierMixin, BaseEstimator):
    """
    Stage 1: 6-class LR base.
    Stage 2: binary DS/Grok specialist, triggered when DS+Grok are the top-2 classes.
    """
    _estimator_type = 'classifier'
    DEEPSEEK = 1
    GROK     = 2

    def __init__(self, base_classifier=None, binary_classifier=None,
                 margin_trigger_gap=None, binary_ds_threshold=0.5, top2_trigger=False):
        self.base_classifier    = base_classifier
        self.binary_classifier  = binary_classifier
        self.margin_trigger_gap = margin_trigger_gap
        self.binary_ds_threshold = binary_ds_threshold
        self.top2_trigger       = top2_trigger

    def fit(self, X, y):
        self.classes_  = np.unique(y)
        self.base_clf_ = clone(self.base_classifier) if self.base_classifier is not None else \
            LogisticRegression(C=0.5, max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=42)
        self.base_clf_.fit(X, y)
        mask = (y == self.DEEPSEEK) | (y == self.GROK)
        if mask.sum() < 4:
            self.binary_clf_ = None
            return self
        self.binary_clf_ = clone(self.binary_classifier) if self.binary_classifier is not None else \
            LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', class_weight=None, random_state=42)
        self.binary_clf_.fit(X[mask], y[mask])
        return self

    def _trigger_mask(self, X, preds, proba=None):
        base_ambig = (preds == self.DEEPSEEK) | (preds == self.GROK)
        if proba is None:
            return base_ambig
        conditions = [base_ambig]
        if self.top2_trigger:
            sorted_idx = np.argsort(proba, axis=1)[:, -2:]
            conditions.append(np.all(np.isin(sorted_idx, [self.DEEPSEEK, self.GROK]), axis=1))
        if self.margin_trigger_gap is not None:
            margin = np.abs(proba[:, self.DEEPSEEK] - proba[:, self.GROK])
            conditions.append(margin < self.margin_trigger_gap)
        result = conditions[0]
        for c in conditions[1:]:
            result = result & c
        return result

    def _apply_binary(self, X_sub):
        if not hasattr(self.binary_clf_, 'predict_proba'):
            return self.binary_clf_.predict(X_sub)
        bp  = self.binary_clf_.predict_proba(X_sub)
        bc  = self.binary_clf_.classes_
        if self.DEEPSEEK not in bc or self.GROK not in bc:
            return self.binary_clf_.predict(X_sub)
        ds_pos = int(np.where(bc == self.DEEPSEEK)[0][0])
        return np.where(bp[:, ds_pos] >= self.binary_ds_threshold, self.DEEPSEEK, self.GROK)

    def predict(self, X):
        need_proba = self.margin_trigger_gap is not None or self.top2_trigger
        if hasattr(self.base_clf_, 'predict_proba') and need_proba:
            proba = self.base_clf_.predict_proba(X)
            preds = np.argmax(proba, axis=1).copy()
        else:
            proba = None
            preds = self.base_clf_.predict(X).copy()
        if self.binary_clf_ is None:
            return preds
        mask = self._trigger_mask(X, preds, proba)
        if mask.any():
            preds[mask] = self._apply_binary(X[mask])
        return preds

    def predict_proba(self, X):
        proba = self.base_clf_.predict_proba(X).copy()
        if self.binary_clf_ is None or not hasattr(self.binary_clf_, 'predict_proba'):
            return proba
        preds = np.argmax(proba, axis=1)
        mask  = self._trigger_mask(X, preds, proba)
        if mask.any():
            bp  = self.binary_clf_.predict_proba(X[mask])
            bc  = self.binary_clf_.classes_
            if self.DEEPSEEK in bc and self.GROK in bc:
                ds_pos = int(np.where(bc == self.DEEPSEEK)[0][0])
                gk_pos = int(np.where(bc == self.GROK)[0][0])
                for i, idx in enumerate(np.where(mask)[0]):
                    total = proba[idx, self.DEEPSEEK] + proba[idx, self.GROK]
                    if total > 0:
                        proba[idx, self.DEEPSEEK] = total * bp[i, ds_pos]
                        proba[idx, self.GROK]     = total * bp[i, gk_pos]
        return proba


class LGBMTfidfClassifier(ClassifierMixin, BaseEstimator):
    """TruncatedSVD(n) + LightGBM with balanced sample weights."""
    _estimator_type = 'classifier'

    def __init__(self, n_svd_components=300, n_estimators=300, num_leaves=31,
                 learning_rate=0.05, min_child_samples=10, subsample=0.8,
                 colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, random_state=42):
        self.n_svd_components  = n_svd_components
        self.n_estimators      = n_estimators
        self.num_leaves        = num_leaves
        self.learning_rate     = learning_rate
        self.min_child_samples = min_child_samples
        self.subsample         = subsample
        self.colsample_bytree  = colsample_bytree
        self.reg_alpha         = reg_alpha
        self.reg_lambda        = reg_lambda
        self.random_state      = random_state

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.svd_     = TruncatedSVD(n_components=self.n_svd_components, random_state=self.random_state)
        X_dense       = self.svd_.fit_transform(X)
        cw = compute_class_weight('balanced', classes=self.classes_, y=y)
        sw = np.array([cw[int(np.where(self.classes_ == yi)[0][0])] for yi in y])
        self.lgbm_    = LGBMClassifier(
            n_estimators=self.n_estimators, num_leaves=self.num_leaves,
            learning_rate=self.learning_rate, min_child_samples=self.min_child_samples,
            subsample=self.subsample, colsample_bytree=self.colsample_bytree,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            random_state=self.random_state, n_jobs=-1, verbose=-1,
        )
        self.lgbm_.fit(X_dense, y, sample_weight=sw)
        return self

    def predict(self, X):       return self.lgbm_.predict(self.svd_.transform(X))
    def predict_proba(self, X): return self.lgbm_.predict_proba(self.svd_.transform(X))


print('Model classes defined.')

## 3. Load Pre-trained Model & Thresholds

In [ ]:
import time

# ── Load model ────────────────────────────────────────────────────────────────
print(f'Loading model from: {MODEL_FILE}')
t0 = time.time()
pipeline = joblib.load(MODEL_FILE)
print(f'Model loaded in {time.time()-t0:.1f}s')
print(f'Pipeline steps: {[name for name, _ in pipeline.steps]}')

# ── Load thresholds ───────────────────────────────────────────────────────────
with open(THRESHOLDS_FILE, 'r') as f:
    thresh_data = json.load(f)

THRESHOLDS         = np.array(thresh_data['thresholds'])
DS_GROK_THRESHOLD  = thresh_data['ds_grok_pair_threshold']
MODEL_NAME         = thresh_data['model']

print(f'\nModel name        : {MODEL_NAME}')
print(f'Per-class thresholds: {THRESHOLDS.tolist()}')
print(f'DS/Grok pair threshold: {DS_GROK_THRESHOLD}')

## 4. Load Test Data

In [ ]:
test_df = pd.read_csv(TEST_FILE)
print(f'Test shape   : {test_df.shape}')
print(f'Test columns : {test_df.columns.tolist()}')

# Drop unnamed index columns if present
unnamed_cols = [c for c in test_df.columns if 'Unnamed' in str(c)]
if unnamed_cols:
    test_df = test_df.drop(columns=unnamed_cols)
    print(f'Dropped: {unnamed_cols}')

text_col = 'TEXT' if 'TEXT' in test_df.columns else test_df.columns[0]
X_test   = test_df[text_col].fillna('').astype(str).tolist()
print(f'Test samples : {len(X_test)}')
print(f'Sample text  : {X_test[0][:120]}...')

## 5. Threshold Functions

In [ ]:
def apply_thresholds(proba, thresholds):
    """Scale class probabilities by per-class factors, then argmax."""
    return np.argmax(proba * thresholds, axis=1)


def apply_ds_grok_pair_threshold(proba, preds, pair_threshold):
    """
    For samples where DS and Grok are the top candidates, apply a ratio threshold.
    ratio = P(DS) / (P(DS) + P(Grok)); predict DS when ratio >= pair_threshold.
    Thresholds > 0.5 are conservative (bias toward Grok).
    """
    DS, GK = 1, 2
    preds  = preds.copy()
    ds_sum = proba[:, DS] + proba[:, GK]
    ambig  = (ds_sum > 0.15) & ((preds == DS) | (preds == GK))
    if not ambig.any():
        return preds
    ratio = proba[ambig, DS] / np.maximum(ds_sum[ambig], 1e-9)
    preds[ambig] = np.where(ratio >= pair_threshold, DS, GK)
    return preds

print('Threshold functions defined.')

## 6. Generate Predictions

In [ ]:
print(f'Running inference on {len(X_test)} test samples...')
t0 = time.time()

# Get probabilities from the pre-trained stacking pipeline
test_proba = pipeline.predict_proba(X_test)
print(f'Inference done in {time.time()-t0:.1f}s  |  proba shape: {test_proba.shape}')

# Apply per-class thresholds
test_preds = apply_thresholds(test_proba, THRESHOLDS)

# Apply DS/Grok pair threshold
test_preds = apply_ds_grok_pair_threshold(test_proba, test_preds, DS_GROK_THRESHOLD)

# Distribution
CLASS_NAMES = ['Human', 'DeepSeek', 'Grok', 'Claude', 'Gemini', 'ChatGPT']
unique, counts = np.unique(test_preds, return_counts=True)
print('\nPrediction distribution:')
for cls, cnt in zip(unique, counts):
    pct = cnt / len(test_preds) * 100
    print(f'  {cls} ({CLASS_NAMES[cls]:8s}): {cnt:3d} ({pct:.1f}%)')

## 7. Save Submission

In [ ]:
# Build submission
submission_df = pd.DataFrame({
    'id':    range(len(test_preds)),
    'label': test_preds.astype(int),
})

# Validate against sample submission
if SAMPLE_FILE.exists():
    sample = pd.read_csv(SAMPLE_FILE)
    unnamed = [c for c in sample.columns if 'Unnamed' in str(c)]
    if unnamed:
        sample = sample.drop(columns=unnamed)
    assert len(submission_df) == len(sample), \
        f'Row count mismatch: {len(submission_df)} vs {len(sample)}'
    print(f'Validation OK: {len(submission_df)} rows match sample submission.')

# Save
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path  = OUTPUT_DIR / f'submission_{timestamp}.csv'
submission_df.to_csv(out_path, index=False)
print(f'Submission saved: {out_path}')

submission_df.head(10)

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=' * 60)
print('SUBMISSION SUMMARY')
print('=' * 60)
print(f'Run          : 2026-03-16_221821_run')
print(f'Model        : {MODEL_NAME}  (stacking_lgbm)')
print(f'CV F1        : 0.9393  (mean of 5 folds)')
print(f'OOF F1       : 0.9397')
print(f'LB score     : 0.92422')
print(f'Thresholds   : {THRESHOLDS.tolist()}')
print(f'DS/Grok thr  : {DS_GROK_THRESHOLD}')
print(f'Predictions  : {len(test_preds)}')
print(f'Submission   : {out_path}')
print('=' * 60)